[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/05_change_detection.ipynb)

# Notebook 05: Change Detection

Loads the multi-temporal stacks built by notebook 04, computes log-ratio delta stacks
(each epoch minus baseline), estimates noise sigma from a stable reference polygon,
generates binary change masks, and exports zonal statistics timeseries.

All thresholds, epoch lists, CRS, baseline year, and AOI paths come from `config.yaml`
— no hardcoded values in this notebook.

**Prerequisites:** Run notebook 04 (`04_build_stacks.ipynb`) first.

**Outputs:**
- `data/processed/{sensor}_{pol}_delta.nc` — log-ratio delta stack (one per sensor × pol)
- `data/figures/{sensor}_{pol}_mask_{year}.tif` — binary change mask GeoTIFF per epoch
- `data/stats/05_sigma_thresholds.json` — sigma and threshold values for audit trail
- `outputs/figures/05_*` — preview maps and timeseries plots

---
### Sigma estimation — which approach to use?

**Approach A (baseline-only):** Sigma estimated from the baseline epoch alone within
the stable reference polygon. Lower sigma → more sensitive, more potential false positives.
Recommended for the first run.

**Approach B (temporal):** Sigma pooled across all epochs in the stable reference. More
conservative because it includes natural inter-annual variability in the noise estimate.
Use if Approach A produces too many false positives in visually stable areas.

Switch approaches by changing `active_sigmas` in Cell 5.

## Setup

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rioxarray', 'netcdf4', 'pyogrio', 'rasterio'], check=True)

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs
import pipeline.change as change_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE       = cfg['site_name']
OUT_FIGS   = OUT / 'figures'
STACKS_DIR = DATA / 'stacks'
ROOT       = repo_root()

aoi_path = ROOT / cfg['aoi']['change_area']
ref_path = ROOT / cfg['aoi']['stable_reference']
crs      = resolve_crs(cfg, aoi_path)

all_years  = [e['year'] for e in cfg['epochs']]
baseline   = cfg['change_detection']['baseline_year']
multiplier = float(cfg['change_detection']['sigma_threshold_multiplier'])
pols       = cfg['opera'].get('polarizations') or ['HH', 'HV']

print(f'Site           : {SITE}')
print(f'Baseline year  : {baseline}')
print(f'Sigma mult     : {multiplier}')
print(f'Polarizations  : {pols}')
print(f'Output CRS     : {crs}')
print(f'Stacks dir     : {STACKS_DIR}')
print(f'Stable ref     : {ref_path}')

## Load multi-temporal stacks

Reads the NetCDF stacks produced by notebook 04. Missing stacks are skipped with a
warning — change detection will proceed for whichever sensors are available.

In [ ]:
# ── Cell 2: load stacks from data/stacks/ ─────────────────────────────────────────
opera_stacks = {}
hyp3_stacks  = {}

for pol in pols:
    opera_path = STACKS_DIR / f'opera_{pol}.nc'
    hyp3_path  = STACKS_DIR / f'hyp3_{pol}.nc'

    if opera_path.exists():
        opera_stacks[pol] = xr.open_dataarray(opera_path, engine='netcdf4')
        epochs_loaded = [pd.Timestamp(t).year for t in opera_stacks[pol].time.values]
        print(f'OPERA {pol}: {opera_stacks[pol].shape}  epochs={epochs_loaded}')
    else:
        print(f'OPERA {pol}: NOT FOUND — {opera_path.name}  (run notebook 04 first)')

    if hyp3_path.exists():
        hyp3_stacks[pol] = xr.open_dataarray(hyp3_path, engine='netcdf4')
        epochs_loaded = [pd.Timestamp(t).year for t in hyp3_stacks[pol].time.values]
        print(f'HyP3  {pol}: {hyp3_stacks[pol].shape}  epochs={epochs_loaded}')
    else:
        print(f'HyP3  {pol}: NOT FOUND — {hyp3_path.name}  (run notebook 04 first)')

# S2 at OPERA grid preferred; fall back to native S2 grid
s2_stack = None
for s2_candidate in ['s2_at_opera.nc', 's2_at_hyp3.nc', 's2_indices.nc']:
    s2_path = STACKS_DIR / s2_candidate
    if s2_path.exists():
        s2_stack = xr.open_dataset(s2_path, engine='netcdf4')
        print(f'S2 ({s2_candidate}): dims={dict(s2_stack.dims)}')
        break
if s2_stack is None:
    print('S2 stack: NOT FOUND — optical delta will be skipped')

if not opera_stacks and not hyp3_stacks:
    raise RuntimeError('No SAR stacks found. Run notebook 04 first.')

## Compute SAR delta stacks

For each epoch: `delta = epoch_dB - baseline_dB`  
Equivalent to a log-ratio in linear power space. The baseline epoch has delta = 0 by
definition and is retained in the stack. Positive values = backscatter increase;
negative = decrease.

In [ ]:
# ── Cell 3: compute SAR delta stacks ──────────────────────────────────────────────
delta_opera = {}
delta_hyp3  = {}

for pol, da in opera_stacks.items():
    stack_years = [pd.Timestamp(t).year for t in da.time.values]
    if baseline not in stack_years:
        print(f'WARNING: baseline {baseline} not in OPERA {pol} stack ({stack_years}). '
              f'compute_delta will use nearest epoch as baseline.')
    delta_opera[pol] = change_lib.compute_delta(da, baseline)
    non_baseline = [y for y in stack_years if y != baseline]
    print(f'OPERA delta {pol}: baseline={baseline}  post-baseline epochs={non_baseline}')

for pol, da in hyp3_stacks.items():
    stack_years = [pd.Timestamp(t).year for t in da.time.values]
    if baseline not in stack_years:
        print(f'WARNING: baseline {baseline} not in HyP3 {pol} stack ({stack_years}). '
              f'compute_delta will use nearest epoch as baseline.')
    delta_hyp3[pol] = change_lib.compute_delta(da, baseline)
    non_baseline = [y for y in stack_years if y != baseline]
    print(f'HyP3  delta {pol}: baseline={baseline}  post-baseline epochs={non_baseline}')

## Compute Sentinel-2 delta (auxiliary)

Applies the same baseline subtraction to each spectral index. Optical deltas are used
as auxiliary evidence alongside SAR — they are not used for the primary change threshold.

In [ ]:
# ── Cell 4: compute S2 spectral-index delta (auxiliary) ───────────────────────────
delta_s2 = None

if s2_stack is None:
    print('S2 stack not loaded — skipping optical delta.')
else:
    s2_years = [pd.Timestamp(t).year for t in s2_stack.time.values]
    if baseline not in s2_years:
        print(f'WARNING: baseline {baseline} not in S2 stack ({s2_years}). '
              f'compute_delta will use nearest epoch.')

    # compute_delta is designed for DataArrays; apply to each Dataset variable
    delta_s2 = xr.Dataset({
        var: change_lib.compute_delta(s2_stack[var], baseline)
        for var in s2_stack.data_vars
    })
    print(f'S2 delta: {list(delta_s2.data_vars)}  baseline={baseline}')
    for var in delta_s2.data_vars:
        arr = delta_s2[var].values
        print(f'  {var}: range=[{np.nanmin(arr):.3f}, {np.nanmax(arr):.3f}]')

## Estimate noise sigma from stable reference

The stable reference polygon (`aoi/stable_reference.geojson`) should cover a surface
that is known to be stable over the entire monitoring period (e.g. bedrock, open water,
bare soil away from the change area). Sigma is the standard deviation of backscatter
within that polygon.

Change is flagged where `|delta| > multiplier × sigma` (from `config.yaml`).

In [ ]:
# ── Cell 5: estimate sigma from stable reference polygon ──────────────────────────
sigmas_a = {}   # Approach A: baseline epoch only
sigmas_b = {}   # Approach B: temporal (all epochs pooled)

for pol, da in opera_stacks.items():
    key = f'opera_{pol}'
    try:
        sigmas_a[key] = change_lib.sigma_from_baseline(da, baseline, ref_path, crs)
        sigmas_b[key] = change_lib.sigma_temporal(da, ref_path, crs)
        print(f'OPERA {pol}:  '
              f'σ_A={sigmas_a[key]:.3f} dB  thresh_A={sigmas_a[key] * multiplier:.2f} dB  |  '
              f'σ_B={sigmas_b[key]:.3f} dB  thresh_B={sigmas_b[key] * multiplier:.2f} dB')
    except ValueError as e:
        print(f'OPERA {pol}: sigma estimation failed — {e}')

for pol, da in hyp3_stacks.items():
    key = f'hyp3_{pol}'
    try:
        sigmas_a[key] = change_lib.sigma_from_baseline(da, baseline, ref_path, crs)
        sigmas_b[key] = change_lib.sigma_temporal(da, ref_path, crs)
        print(f'HyP3  {pol}:  '
              f'σ_A={sigmas_a[key]:.3f} dB  thresh_A={sigmas_a[key] * multiplier:.2f} dB  |  '
              f'σ_B={sigmas_b[key]:.3f} dB  thresh_B={sigmas_b[key] * multiplier:.2f} dB')
    except ValueError as e:
        print(f'HyP3  {pol}: sigma estimation failed — {e}')

# ── Choose active approach ──────────────────────────────────────────────────────────
# Switch to sigmas_b for the more conservative temporal estimate.
active_sigmas = sigmas_a
print(f'\nActive sigma approach: A (baseline-only)  '
      f'— change active_sigmas to sigmas_b to use temporal approach.')

## Generate binary change masks

For each pixel and epoch: `1` if `|delta| > sigma × multiplier`, `0` otherwise, `NaN` if no data.  
The threshold is set by `change_detection.sigma_threshold_multiplier` in `config.yaml`.

In [ ]:
# ── Cell 6: generate binary change masks ──────────────────────────────────────────
masks_opera = {}
masks_hyp3  = {}

for pol, delta in delta_opera.items():
    key   = f'opera_{pol}'
    sigma = active_sigmas.get(key)
    if sigma is None:
        print(f'OPERA {pol}: no sigma available — skipping mask')
        continue
    masks_opera[pol] = change_lib.make_changemask(delta, sigma, multiplier)
    pct = float(np.nanmean(masks_opera[pol].values)) * 100
    print(f'OPERA {pol}: threshold={sigma * multiplier:.2f} dB  '
          f'~{pct:.1f}% of pixel-epochs flagged')

for pol, delta in delta_hyp3.items():
    key   = f'hyp3_{pol}'
    sigma = active_sigmas.get(key)
    if sigma is None:
        print(f'HyP3  {pol}: no sigma available — skipping mask')
        continue
    masks_hyp3[pol] = change_lib.make_changemask(delta, sigma, multiplier)
    pct = float(np.nanmean(masks_hyp3[pol].values)) * 100
    print(f'HyP3  {pol}: threshold={sigma * multiplier:.2f} dB  '
          f'~{pct:.1f}% of pixel-epochs flagged')

## Zonal statistics timeseries

Per-epoch mean backscatter (dB) and standard deviation within the change area and stable
reference polygons. Useful for quantifying the magnitude and direction of change.

In [ ]:
# ── Cell 7: compute zonal statistics timeseries ───────────────────────────────────
aoi_polygons = {
    'change_area'      : aoi_path,
    'stable_reference' : ref_path,
}

df_opera_ts = None
df_hyp3_ts  = None

if opera_stacks:
    pol = pols[0]
    df_opera_ts = change_lib.zonal_stats_timeseries(opera_stacks[pol], aoi_polygons, crs)
    df_opera_ts['sensor'] = f'OPERA {pol}'
    print(f'OPERA {pol} zonal stats:')
    print(df_opera_ts.to_string(index=False))
    print()

if hyp3_stacks:
    pol = pols[0]
    df_hyp3_ts = change_lib.zonal_stats_timeseries(hyp3_stacks[pol], aoi_polygons, crs)
    df_hyp3_ts['sensor'] = f'HyP3 {pol}'
    print(f'HyP3 {pol} zonal stats:')
    print(df_hyp3_ts.to_string(index=False))

## Save outputs

- Delta stacks → `data/processed/`
- Change mask GeoTIFFs → `data/figures/` (one per sensor × pol × epoch)
- Sigma report → `data/stats/`

In [ ]:
# ── Cell 8: save delta stacks, change masks, and sigma report ─────────────────────
# Delta stacks
for pol, delta in delta_opera.items():
    path = change_lib.save_delta_stack(delta, DATA, f'opera_{pol}')
    print(f'Saved delta: {path.name}')

for pol, delta in delta_hyp3.items():
    path = change_lib.save_delta_stack(delta, DATA, f'hyp3_{pol}')
    print(f'Saved delta: {path.name}')

# Change mask GeoTIFFs — one file per pol × epoch
mask_count = 0
for pol, mask_stack in masks_opera.items():
    for t in mask_stack.time.values:
        year = pd.Timestamp(t).year
        change_lib.save_changemask(mask_stack.sel(time=t), DATA, f'opera_{pol}', year)
        mask_count += 1

for pol, mask_stack in masks_hyp3.items():
    for t in mask_stack.time.values:
        year = pd.Timestamp(t).year
        change_lib.save_changemask(mask_stack.sel(time=t), DATA, f'hyp3_{pol}', year)
        mask_count += 1

print(f'Saved {mask_count} change mask GeoTIFF(s)')

# Sigma report (JSON — full audit trail)
sigma_report = {
    'site'                  : SITE,
    'baseline_year'         : baseline,
    'sigma_multiplier'      : multiplier,
    'approach_a_baseline'   : sigmas_a,
    'approach_b_temporal'   : sigmas_b,
    'active_approach'       : 'a_baseline_only',
}
path = change_lib.save_sigma_report(sigma_report, DATA)
print(f'Saved sigma report: {path.name}')

## Preview — change maps per epoch

Red = backscatter increase; blue = decrease. Only pixels that exceed the change
threshold are coloured; stable pixels are masked out.

In [ ]:
# ── Cell 9: plot signed change maps ───────────────────────────────────────────────
masks  = masks_opera  if masks_opera  else masks_hyp3
deltas = delta_opera  if masks_opera  else delta_hyp3
sensor_label = 'OPERA' if masks_opera else 'HyP3'

if not masks:
    print('No change masks available — run cells above first.')
else:
    pol         = pols[0]
    mask_stack  = masks.get(pol)
    delta_stack = deltas.get(pol)

    if mask_stack is None or delta_stack is None:
        print(f'Mask or delta not available for {sensor_label} {pol}.')
    else:
        n_epochs = len(mask_stack.time)
        fig, axes = plt.subplots(1, n_epochs, figsize=(5 * n_epochs, 4), squeeze=False)
        axes = axes[0]

        for ax, t in zip(axes, mask_stack.time.values):
            year         = pd.Timestamp(t).year
            delta_slice  = delta_stack.sel(time=t)
            mask_slice   = mask_stack.sel(time=t)

            # colour only pixels that crossed the change threshold
            signed = delta_slice.where(mask_slice > 0)
            vals   = signed.values[np.isfinite(signed.values)]
            if vals.size > 0:
                vmax = max(abs(np.percentile(vals, 2)), abs(np.percentile(vals, 98)))
                vmax = vmax if vmax > 0 else 3.0
            else:
                vmax = 3.0

            ax.imshow(signed.values, cmap='RdBu', vmin=-vmax, vmax=vmax)
            title = f'{year} vs {baseline}\n(no change)' if vals.size == 0 else f'{year} vs {baseline}'
            ax.set_title(f'{sensor_label} {pol}\n{title}')
            ax.axis('off')

        plt.suptitle(f'{SITE} — SAR change detection (σ×{multiplier})', fontsize=11)
        plt.tight_layout()
        OUT_FIGS.mkdir(parents=True, exist_ok=True)
        save_path = OUT_FIGS / f'05_{sensor_label.lower()}_{pol}_change_maps.png'
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {save_path}')

## Backscatter timeseries — change area vs. stable reference

Line plot of mean backscatter per epoch within each polygon. The gap between the
change-area and stable-reference lines indicates the change signal magnitude.

In [ ]:
# ── Cell 10: plot zonal backscatter timeseries ─────────────────────────────────────
df_ts       = df_opera_ts if df_opera_ts is not None else df_hyp3_ts
sensor_plot = 'OPERA' if df_opera_ts is not None else 'HyP3'

if df_ts is None:
    print('No zonal statistics available — run Cell 7 first.')
else:
    pol = pols[0]
    fig, ax = plt.subplots(figsize=(10, 4))

    colors = {'change_area': '#d62728', 'stable_reference': '#1f77b4'}
    for aoi_label, group in df_ts.groupby('aoi'):
        color = colors.get(aoi_label, None)
        ax.plot(group['year'], group['mean'], marker='o',
                label=aoi_label.replace('_', ' '), color=color)
        ax.fill_between(
            group['year'],
            group['mean'] - group['std'],
            group['mean'] + group['std'],
            alpha=0.15, color=color,
        )

    ax.axvline(baseline, color='gray', ls='--', lw=1, label=f'baseline ({baseline})')
    ax.set_xlabel('Year')
    ax.set_ylabel('Backscatter (dB)')
    ax.set_title(f'{SITE} — {sensor_plot} {pol} mean backscatter timeseries')
    ax.legend()
    plt.tight_layout()

    OUT_FIGS.mkdir(parents=True, exist_ok=True)
    save_path = OUT_FIGS / f'05_{pol.lower()}_backscatter_timeseries.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')